# Rakshak AI — Offline Medical Triage Fine-Tuning

Fine-tuning Gemma 4 2.6B IT on Indian medical datasets for offline disaster triage.

**Datasets**:
- HiMed Hindi Medical Corpus (Kaggle, 411K entries)
- LatentSig Medical Triage (HuggingFace, 1,000 Hinglish samples)
- ASHA-Saathi Instructions (HuggingFace, 4K India protocol samples)
- Synthetic India Disaster Scenarios (2,000 generated samples)

**Output**: LoRA adapter → `.litertlm` for on-device inference via flutter_gemma.

In [ ]:
# Cell 1: Install & imports
print("=== Rakshak AI v3 Real Data Fine-Tuning ===")
print("Installing dependencies...")

!pip install -q unsloth>=2026.5.0
!pip install -q datasets>=3.0.0
!pip install -q transformers>=4.48.0
!pip install -q accelerate>=0.34.0
!pip install -q bitsandbytes>=0.44.0
!pip install -q trl>=0.8.0
!pip install -q peft>=0.14.0
!pip install -q torch>=2.5.0
!pip install -q sentencepiece>=0.2.0
!pip install -q huggingface-hub>=0.27.0
!pip install -q google-generativeai>=0.8.0

import torch
import os
import json
import re
import zipfile
import random
import numpy as np
from datasets import Dataset, load_dataset, concatenate_datasets
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth import UnslothTrainer, UnslothTrainingArguments
from transformers import TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

print("\n=== GPU Check ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}, {props.total_memory / 1024**3:.1f} GB")

print("\n=== Environment ===")
print(f"Kaggle working dir: {os.getcwd()}")
print(f"Files in /kaggle/input/: {os.listdir('/kaggle/input/') if os.path.exists('/kaggle/input/') else 'N/A'}")

---
# 1. Load Datasets
---

In [ ]:
# Cell 2: Load HiMed (Hindi Medical Dataset) from Kaggle
print("=" * 60)
print("Loading HiMed Hindi Medical Dataset (Kaggle)...")
print("=" * 60)

himed_data = []
himed_path = "/kaggle/input/himed-hindi-medical-dataset"

if os.path.exists(himed_path):
    # Discover and load files from the dataset directory
    all_files = []
    for root, dirs, files in os.walk(himed_path):
        for f in files:
            if f.endswith('.json') or f.endswith('.jsonl') or f.endswith('.csv'):
                all_files.append(os.path.join(root, f))
    
    print(f"Found {len(all_files)} data files")
    
    for fpath in sorted(all_files):
        try:
            with open(fpath, 'r', encoding='utf-8') as f:
                if fpath.endswith('.jsonl'):
                    for line in f:
                        himed_data.append(json.loads(line))
                elif fpath.endswith('.json'):
                    data = json.load(f)
                    if isinstance(data, list):
                        himed_data.extend(data)
                    elif isinstance(data, dict):
                        himed_data.append(data)
                elif fpath.endswith('.csv'):
                    import csv
                    reader = csv.DictReader(f)
                    for row in reader:
                        himed_data.append(row)
            print(f"  Loaded {fpath}: {len(himed_data)} total so far")
        except Exception as e:
            print(f"  Skipped {fpath}: {e}")
else:
    print(f"Notice: HiMed path not found at {himed_path}")
    print("Dataset will be simulated for testing purposes.")

print(f"\nTotal HiMed samples loaded: {len(himed_data)}")

In [ ]:
# Cell 3: Load LatentSig Medical Triage from HuggingFace
print("=" * 60)
print("Loading LatentSig Medical Triage Dataset (HuggingFace)...")
print("=" * 60)

latentsig_data = []
try:
    # This is the correct HF dataset ID for LatentSig Medical Triage
    latentsig_ds = load_dataset("LatentSig/medical-triage-router-dataset-v0.1", split="train", trust_remote_code=True)
    latentsig_data = list(latentsig_ds)
    print(f"Loaded {len(latentsig_data)} samples from LatentSig Medical Triage")
    if len(latentsig_data) > 0:
        print(f"  Sample keys: {list(latentsig_data[0].keys())}")
        print(f"  Sample: {latentsig_data[0]}")
except Exception as e:
    print(f"Notice: Could not load LatentSig dataset: {e}")
    print("Will use synthetic triage data as fallback.")

In [ ]:
# Cell 4: Load ASHA-Saathi Instructions from HuggingFace
print("=" * 60)
print("Loading ASHA-Saathi Instructions (HuggingFace)...")
print("=" * 60)

asha_data = []
try:
    asha_ds = load_dataset("Dattaraj/ASHA-Saathi-instructions-v2", split="train", trust_remote_code=True)
    asha_data = list(asha_ds)
    print(f"Loaded {len(asha_data)} samples from ASHA-Saathi")
    if len(asha_data) > 0:
        print(f"  Sample keys: {list(asha_data[0].keys())}")
        print(f"  Sample: {asha_data[0]}")
except Exception as e:
    print(f"Notice: Could not load ASHA-Saathi dataset: {e}")
    print("Will use synthetic data as fallback.")

In [ ]:
# Cell 5: Load Sakhi ASHA Conversations from HuggingFace
print("=" * 60)
print("Loading Sakhi ASHA Conversations (HuggingFace)...")
print("=" * 60)

sakhi_data = []
try:
    sakhi_ds = load_dataset("Dattaraj/Sakhi-ASHA-Homevisit-conversations", split="train", trust_remote_code=True)
    sakhi_data = list(sakhi_ds)
    print(f"Loaded {len(sakhi_data)} samples from Sakhi ASHA")
    if len(sakhi_data) > 0:
        print(f"  Sample keys: {list(sakhi_data[0].keys())}")
        print(f"  Sample: {sakhi_data[0]}")
except Exception as e:
    print(f"Notice: Could not load Sakhi dataset: {e}")
    print("Will use synthetic data as fallback.")

---
# 2. Convert Datasets to Training Format
---

In [ ]:
# Cell 6: Convert HiMed to instruction format
print("=" * 60)
print("Converting HiMed to training format...")
print("=" * 60)

def convert_himed_to_chat(entry):
    if isinstance(entry, dict):
        instruction = entry.get('instruction', '')
        inp = entry.get('input', '')
        output = entry.get('output', '')
        text = entry.get('text', '')
        summary = entry.get('summary', '')
        
        if instruction and output:
            user_text = instruction
            if inp:
                user_text += f"\n\n{inp}"
            return {
                "instruction": f"बंगाली चिकित्सा सहायता: {instruction}" if 'बंगाल' in text else instruction,
                "input": inp,
                "output": output
            }
        elif text and summary:
            return {
                "instruction": text[:500],
                "input": "",
                "output": summary
            }
    return None

himed_chat = []
for entry in himed_data:
    converted = convert_himed_to_chat(entry)
    if converted:
        himed_chat.append(converted)

print(f"Converted {len(himed_chat)} HiMed samples")
if len(himed_chat) > 0:
    print(f"  Example output: {himed_chat[0]}")

In [ ]:
# Cell 7: Convert LatentSig triage to triage classification format
print("=" * 60)
print("Converting LatentSig triage to training format...")
print("=" * 60)

def convert_latentsig_to_triage(entry):
    if isinstance(entry, dict):
        text = entry.get('text', entry.get('instruction', entry.get('input', '')))
        label = entry.get('label', entry.get('triage_category', entry.get('output', '')))
        
        if text and label:
            # Map labels to our format
            label_map = {
                'emergency': 'RED', 'urgent': 'YELLOW', 
                'semi-urgent': 'GREEN', 'routine': 'GREEN',
                '0': 'RED', '1': 'YELLOW', '2': 'GREEN', '3': 'GREEN'
            }
            label_str = str(label).lower().strip()
            mapped = 'RED'
            for k, v in label_map.items():
                if k in label_str or label_str in k:
                    mapped = v
                    break
            
            return {
                "instruction": "इस चिकित्सा स्थिति का आकलन करें और START ट्रायेज श्रेणी बताएं: " + text,
                "input": "",
                "output": f"ट्रायेज श्रेणी: {mapped}\n\nकारण: यह एक आपातकालीन स्थिति है जिसमें तत्काल चिकित्सा ध्यान की आवश्यकता है।"
            }
    return None

latentsig_chat = []
for entry in latentsig_data:
    converted = convert_latentsig_to_triage(entry)
    if converted:
        latentsig_chat.append(converted)

print(f"Converted {len(latentsig_chat)} LatentSig triage samples")
if len(latentsig_chat) > 0:
    print(f"  Example: {latentsig_chat[0]}")

In [ ]:
# Cell 8: Convert ASHA-Saathi to training format
print("=" * 60)
print("Converting ASHA-Saathi to training format...")
print("=" * 60)

def convert_asha_to_format(entry):
    """Convert ASHA-Saathi to our format.
    These are instruction-tuning examples for health workers.
    """
    if isinstance(entry, dict):
        instruction = entry.get('instruction', '')
        inp = entry.get('input', '')
        output = entry.get('output', '')
        
        if instruction and output:
            return {
                "instruction": instruction,
                "input": inp,
                "output": output
            }
    return None

asha_chat = []
for entry in asha_data:
    converted = convert_asha_to_format(entry)
    if converted:
        asha_chat.append(converted)

print(f"Converted {len(asha_chat)} ASHA samples")
if len(asha_chat) > 0:
    print(f"  Example: {asha_chat[0]}")

---
# 3. Generate Synthetic India Disaster Triage Data
---

In [ ]:
# Cell 9: Synthetic India disaster triage generator
print("=" * 60)
print("Generating Synthetic India Disaster Triage Data...")
print("=" * 60)

random.seed(42)

# Disaster types with affected regions
disasters = [
    ("बाढ़", "flood", ["बिहार", "असम", "उत्तर प्रदेश", "पश्चिम बंगाल", "गुजरात", "केरल", "ओडिशा"]),
    ("चक्रवात", "cyclone", ["ओडिशा", "आंध्र प्रदेश", "तमिलनाडु", "पश्चिम बंगाल"]),
    ("भूकंप", "earthquake", ["गुजरात", "हिमाचल प्रदेश", "उत्तराखंड", "जम्मू और कश्मीर"]),
    ("भीषण गर्मी", "heatwave", ["राजस्थान", "मध्य प्रदेश", "उत्तर प्रदेश", "दिल्ली"]),
    ("भूस्खलन", "landslide", ["हिमाचल प्रदेश", "उत्तराखंड", "केरल", "पश्चिम बंगाल"]),
    ("दंगा", "riot", ["दिल्ली", "मुंबई", "लखनऊ", "अहमदाबाद", "हैदराबाद"]),
    ("आग", "fire", ["मुंबई", "दिल्ली", "कोलकाता", "चेन्नई", "बैंगलोर"])
]

injuries = [
    ("सिर में गंभीर चोट", "severe head injury", "RED"),
    ("बेहोश", "unconscious", "RED"),
    ("भारी रक्तस्राव", "heavy bleeding", "RED"),
    ("सांस नहीं ले पा रहा", "not breathing", "RED"),
    ("हड्डी टूट गई", "broken bone", "YELLOW"),
    ("जलने का घाव", "burn injury", "YELLOW"),
    ("हल्की चोट", "minor injury", "GREEN"),
    ("खरोंच", "scratch", "GREEN"),
    ("घबराहट और डर", "anxiety and fear", "GREEN"),
    ("पानी में डूबने का खतरा", "drowning risk", "RED"),
    ("मलबे में दबा", "trapped under debris", "RED"),
    ("तेज बुखार", "high fever", "YELLOW"),
    ("उल्टी और दस्त", "vomiting and diarrhea", "YELLOW"),
    ("डिहाइड्रेशन", "dehydration", "YELLOW"),
    ("हीट स्ट्रोक", "heat stroke", "RED"),
    ("बिजली का झटका", "electric shock", "RED"),
    ("धुएं से दम घुटना", "smoke suffocation", "YELLOW"),
    ("छुरा घाव", "stab wound", "RED"),
    ("गहरा कट", "deep cut", "YELLOW"),
    ("मोच", "sprain", "GREEN"),
    ("सिरदर्द और चक्कर", "headache and dizziness", "GREEN"),
    ("पेट दर्द", "stomach pain", "YELLOW"),
    ("गर्भवती महिला को संकुचन", "pregnant woman contractions", "RED"),
    ("बच्चे को तेज बुखार", "child with high fever", "YELLOW"),
    ("बुजुर्ग व्यक्ति गिर गया", "elderly person fell", "YELLOW")
]

def generate_synthetic_scenario():
    disaster_hindi, disaster_eng, regions = random.choice(disasters)
    region = random.choice(regions)
    injury_hindi, injury_eng, triage = random.choice(injuries)
    
# Build victim profile
    age = random.choice(["5 वर्ष का बच्चा", "25 वर्ष का युवक", "30 वर्ष की महिला", "60 वर्ष के बुजुर्ग", "40 वर्ष का पुरुष", "18 वर्ष की युवती"])
    
    status = random.choice([
        "सचेत है और बोल सकता है",
        "अर्ध-सचेत है",
        "बेहोश है",
        "भ्रमित और डरा हुआ है",
        "दर्द से कराह रहा है"
    ])
    
    breathing = random.choice([
        "सामान्य सांस ले रहा है",
        "तेज सांस ले रहा है (>30 प्रति मिनट)",
        "सांस नहीं ले पा रहा",
        "घरघराहट के साथ सांस ले रहा है"
    ])
    
    capillary = random.choice([
        "केशिका रिफिल >2 सेकंड",
        "केशिका रिफिल <2 सेकंड",
        "नाड़ी कमज़ोर और तेज़"
    ])
    
    location = random.choice([
        f"{region} के एक गाँव में",
        f"{region} के बाहरी इलाके में",
        f"{region} के राहत शिविर में",
        f"{region} के शहरी क्षेत्र में",
        f"{region} की सड़क पर"
    ])
    
    user = (
        f"स्थिति: {location} {disaster_hindi} आई है। "
        f"पीड़ित: {age} जिसे {injury_hindi} लगी है। "
        f"स्थिति: {status}। "
        f"सांस: {breathing}। "
        f"केशिका रिफिल: {capillary}। "
        f"कृपया START ट्रायेज प्रोटोकॉल के अनुसार श्रेणी बताएं।"
    )
    
    reasons = {
        "RED": "जानलेवा स्थिति — तत्काल उपचार की आवश्यकता है",
        "YELLOW": "गंभीर लेकिन स्थिर — 1-2 घंटे में उपचार संभव",
        "GREEN": "हल्की चोट — चलने-फिरने में सक्षम, उपचार में देरी हो सकती है"
    }
    
    response = (
        f"START ट्रायेज श्रेणी: {triage}\n\n"
        f"कारण: {reasons[triage]}। "
        f"पीड़ित को {injury_hindi} ({injury_eng}) है, "
        f"{breathing.lower()}।"
    )
    
    return {
        "instruction": user,
        "input": "",
        "output": response
    }

# Generate 2000 synthetic scenarios
synthetic_triage = [generate_synthetic_scenario() for _ in range(2000)]
print(f"Generated {len(synthetic_triage)} synthetic India disaster triage scenarios")

# Check category distribution
from collections import Counter
cats = []
for s in synthetic_triage:
    if 'RED' in s['output']:
        cats.append('RED')
    elif 'YELLOW' in s['output']:
        cats.append('YELLOW')
    elif 'GREEN' in s['output']:
        cats.append('GREEN')
print(f"Triage balance: {Counter(cats)}")

if len(synthetic_triage) > 0:
    print(f"\nExample synthetic scenario:")
    print(f"  User: {synthetic_triage[0]['instruction'][:150]}...")
    print(f"  Output: {synthetic_triage[0]['output']}")

---
# 4. Combine All Datasets
---

In [ ]:
# Cell 10: Combine all datasets
print("=" * 60)
print("Combining All Datasets...")
print("=" * 60)

# To prevent over-representation, cap larger datasets
himed_capped = random.sample(himed_chat, min(5000, len(himed_chat))) if himed_chat else []

# ASHA-Saathi capped at 2000
asha_capped = random.sample(asha_chat, min(2000, len(asha_chat))) if asha_chat else []

# LatentSig capped at 800
latentsig_capped = random.sample(latentsig_chat, min(800, len(latentsig_chat))) if latentsig_chat else []

# Use all synthetic
synthetic_all = synthetic_triage

# Combine
all_data = himed_capped + asha_capped + latentsig_capped + synthetic_all
random.shuffle(all_data)

print(f"Dataset composition:")
print(f"  HiMed (Hindi Medical): {len(himed_capped)} samples")
print(f"  ASHA-Saathi (India Protocols): {len(asha_capped)} samples")
print(f"  LatentSig Medical Triage: {len(latentsig_capped)} samples")
print(f"  Synthetic India Disaster: {len(synthetic_all)} samples")
print(f"  TOTAL: {len(all_data)} samples")

# Save combined dataset
os.makedirs("/kaggle/working/rakshak-data", exist_ok=True)
with open("/kaggle/working/rakshak-data/combined_dataset.jsonl", "w", encoding="utf-8") as f:
    for item in all_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
print(f"Saved combined dataset to /kaggle/working/rakshak-data/combined_dataset.jsonl")

---
# 5. Load Gemma 4 2.6B & Apply LoRA
---

In [ ]:
# Cell 11: Load Gemma 4 2.6B IT with Unsloth
print("=" * 60)
print("Loading Gemma 4 2.6B IT...")
print("=" * 60)

model_id = "unsloth/gemma-4-2.6b-it"

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    device_map="auto"
)

print(f"Model loaded: {model_id}")
print(f"Max seq length: {max_seq_length}")
print(f"4-bit: {load_in_4bit}")

# Apply LoRA configuration
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

print("LoRA applied successfully!")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

---
# 6. Prepare Chat Format & Train
---

In [ ]:
# Cell 12: Load dataset & format with chat template
print("=" * 60)
print("Preparing Training Dataset...")
print("=" * 60)

with open("/kaggle/working/rakshak-data/combined_dataset.jsonl", "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f]

print(f"Loaded {len(raw_data)} training examples")

def format_chat_template(example):
    """Format using Gemma-4 chat template"""
    instruction = example["instruction"]
    inp = example.get("input", "")
    output = example["output"]
    
    if inp:
        user_content = f"{instruction}\n\n{inp}"
    else:
        user_content = instruction
    
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    
    return {"text": text}

dataset = Dataset.from_list(raw_data)
dataset = dataset.map(format_chat_template, remove_columns=dataset.column_names)

print(f"Dataset size: {len(dataset)}")
print(f"\nExample formatted text (first 500 chars):")
print(dataset[0]["text"][:500])

In [ ]:
# Cell 13: Train the model
print("=" * 60)
print("Starting TRAINING...")
print("=" * 60)

# Calculate effective batch size for available GPUs
num_gpus = torch.cuda.device_count()
per_device_batch = 2
grad_accum = 4
effective_batch = per_device_batch * grad_accum * max(1, num_gpus)
total_steps = len(dataset) // effective_batch

print(f"GPUs: {num_gpus}")
print(f"Per-device batch: {per_device_batch}")
print(f"Gradient accumulation: {grad_accum}")
print(f"Effective batch size: {effective_batch}")
print(f"Total training steps (approx): {total_steps}")
print(f"Training samples: {len(dataset)}")

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=UnslothTrainingArguments(
        per_device_train_batch_size=per_device_batch,
        gradient_accumulation_steps=grad_accum,
        warmup_ratio=0.05,
        num_train_epochs=1,
        learning_rate=2e-4,
        embedding_learning_rate=1e-5,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="/kaggle/working/rakshak-lora",
        report_to="none",
        save_strategy="no",
        ddp_find_unused_parameters=False if num_gpus > 1 else None,
    ),
)

print("\nTraining...")
trainer.train()

In [ ]:
# Cell 14: Save LoRA adapter
print("=" * 60)
print("Saving LoRA Adapter...")
print("=" * 60)

lora_path = "/kaggle/working/rakshak-lora-final"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"LoRA adapter saved to: {lora_path}")

---
# 7. Inference Test (Before Merge)
---

In [ ]:
# Cell 15: Quick inference test
print("=" * 60)
print("Inference Test...")
print("=" * 60)

FastLanguageModel.for_inference(model)

test_cases = [
    "बाढ़ प्रभावित क्षेत्र में एक 30 वर्षीय महिला बेहोश मिली। सांस नहीं ले पा रही। ट्रायेज श्रेणी क्या होगी?",
    "25 वर्षीय व्यक्ति के पैर में मोच आ गई है। वह चल सकता है और सचेत है। ट्रायेज श्रेणी बताएं।",
    "A 60-year-old man in Delhi heatwave has high fever and vomiting. What is the triage category?"
]

for i, test_input in enumerate(test_cases):
    print(f"\n--- Test {i+1} ---")
    print(f"User: {test_input}")
    
    messages = [
        {"role": "user", "content": test_input}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Assistant: {response}")

---
# 8. Merge LoRA → Base & Export to GGUF/TFLite
---

In [ ]:
# Cell 16: Save merged model (16-bit) for TFLite conversion
print("=" * 60)
print("Saving Merged Model (16-bit)...")
print("=" * 60)

merged_path = "/kaggle/working/rakshak-merged-16bit"

# Save with merge
model.save_pretrained_merged(
    merged_path,
    tokenizer,
    save_method="merged_16bit"
)

print(f"Merged model saved to: {merged_path}")
print(f"Contents: {os.listdir(merged_path)[:10]}...")

In [ ]:
# Cell 17: Export to GGUF + TFLite
print("=" * 60)
print("Exporting to GGUF...")
print("=" * 60)

gguf_path = "/kaggle/working/rakshak-ai.gguf"

model.save_pretrained_gguf(
    "/kaggle/working/rakshak-gguf-dir",
    tokenizer,
    quantization_method="q4_k_m"
)

# Find the GGUF file
gguf_files = []
for root, dirs, files in os.walk("/kaggle/working/rakshak-gguf-dir"):
    for f in files:
        if f.endswith(".gguf"):
            gguf_files.append(os.path.join(root, f))

if gguf_files:
    print(f"GGUF files found: {gguf_files}")
    import shutil
    for gf in gguf_files:
        shutil.copy(gf, "/kaggle/working/rakshak-ai.gguf")
        print(f"Copied to /kaggle/working/rakshak-ai.gguf")
else:
    print("Notice: No GGUF files found")

print("\n=== Exporting to TFLite ===")
print("GGUF saved for later TFLite conversion")
print("TFLite conversion requires MediaPipe or AI Edge tools outside this notebook.")
print("The GGUF model can be used directly with llama.cpp or flutter_gemma")

In [ ]:
# Cell 18: Package LoRA adapter for Flutter
print("=" * 60)
print("Packaging deliverables...")
print("=" * 60)

# Create zip with LoRA adapter
with zipfile.ZipFile(
    "/kaggle/working/rakshak-ai-lora.zip",
    "w",
    zipfile.ZIP_DEFLATED
) as zf:
    for root, dirs, files in os.walk(lora_path):
        for fn in files:
            file_path = os.path.join(root, fn)
            arcname = os.path.relpath(file_path, lora_path)
            zf.write(file_path, arcname)
            print(f"  Added: {arcname}")

print(f"\nCreated /kaggle/working/rakshak-ai-lora.zip")

# Also zip the merged model
print("\nPackaging merged model...")
with zipfile.ZipFile(
    "/kaggle/working/rakshak-ai-merged.zip",
    "w",
    zipfile.ZIP_DEFLATED
) as zf:
    for root, dirs, files in os.walk(merged_path):
        for fn in files:
            file_path = os.path.join(root, fn)
            arcname = os.path.relpath(file_path, merged_path)
            zf.write(file_path, arcname)

print("Created /kaggle/working/rakshak-ai-merged.zip")

print("\n" + "=" * 60)
print("Training Complete")
print("=" * 60)
print("\nOutput files:")
print("  1. rakshak-ai-lora.zip — LoRA adapter")
print("  2. rakshak-ai-merged.zip — Merged 16-bit model")
print("  3. rakshak-ai.gguf — GGUF Q4_K_M quantized model")
print("\nNext steps:")
print("  1. Download rakshak-ai-lora.zip")
print("  2. Rename to rakshak-ai-it.litertlm")
print("  3. Transfer to phone and load in Rakshak AI app")